# ATMS N20: ObsValue vs hofx brightnessTemperature
Compares the `iodanc` and `geode` diag files, one figure per channel (rows: file, columns: ObsValue / hofx). Values are averaged onto a lat/lon grid.

In [ ]:
import glob, os
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt

DIAG_DIR = '/scratch4/NCEPDEV/stmp/Cory.R.Martin/geode_ioda_test_case/run_hofx_atms_23882580/diags'
files = {
    'iodanc': f'{DIAG_DIR}/diag_radiance_atms_n20_iodanc.nc',
    'geode': f'{DIAG_DIR}/diag_radiance_atms_n20_geode.nc',
}
BIN_DEG = 1.0  # grid cell size for averaging

In [ ]:
def clean(a):
    a = np.ma.filled(np.ma.masked_invalid(a).astype('float64'), np.nan)
    a[np.abs(a) > 1e10] = np.nan  # IODA missing-value fill
    return a

data = {}
for name, path in files.items():
    with nc.Dataset(path) as ds:
        data[name] = {
            'lat': clean(ds['MetaData/latitude'][:]),
            'lon': clean(ds['MetaData/longitude'][:]),
            'obs': clean(ds['ObsValue/brightnessTemperature'][:]),
            'hofx': clean(ds['hofx/brightnessTemperature'][:]),
        }
    d = data[name]
    d['lon'] = (d['lon'] + 180) % 360 - 180
    print(name, d['obs'].shape, 'valid obs:', np.isfinite(d['obs']).sum(), 'valid hofx:', np.isfinite(d['hofx']).sum())

In [ ]:
lon_edges = np.arange(-180, 180 + BIN_DEG, BIN_DEG)
lat_edges = np.arange(-90, 90 + BIN_DEG, BIN_DEG)

def grid_mean(lat, lon, val):
    ok = np.isfinite(val) & np.isfinite(lat) & np.isfinite(lon)
    s, _, _ = np.histogram2d(lat[ok], lon[ok], bins=[lat_edges, lon_edges], weights=val[ok])
    n, _, _ = np.histogram2d(lat[ok], lon[ok], bins=[lat_edges, lon_edges])
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(n > 0, s / n, np.nan)

nchan = data['iodanc']['obs'].shape[1]
for ch in range(nchan):
    # shared color range per channel across all four panels
    vals = np.concatenate([data[f][k][:, ch] for f in data for k in ('obs', 'hofx')])
    vals = vals[np.isfinite(vals)]
    vmin, vmax = (np.percentile(vals, [1, 99]) if vals.size else (0, 1))

    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
    for i, f in enumerate(data):
        d = data[f]
        for j, (key, title) in enumerate([('obs', 'ObsValue'), ('hofx', 'hofx')]):
            ax = axes[i, j]
            g = grid_mean(d['lat'], d['lon'], d[key][:, ch])
            m = ax.pcolormesh(lon_edges, lat_edges, g, cmap='turbo', vmin=vmin, vmax=vmax)
            ax.set_title(f'{f}: {title}')
            fig.colorbar(m, ax=ax, label='Tb (K)')
    fig.suptitle(f'ATMS N20 brightnessTemperature, channel {ch + 1}')
    fig.supxlabel('Longitude'); fig.supylabel('Latitude')
    plt.tight_layout()
    plt.show()